In [ ]:
!module load stack/2024-06 gcc/12.2.0 bedtools2/2.31.0

In [ ]:
!pwd

In [ ]:
import os
import pandas as pd
import numpy as np
import glob
import subprocess
from collections import defaultdict

In [ ]:
# Path with files outputted from bedtools intersect
PATH = "/path/to/scratch/data_oxidation/cpg_intersect" 
OUTPUT_FILE = "../data_normalized/cpg_Normalized.csv"
GENOME_PATH = "/path/to/lab_storage/genomes/mouse/GRCm39_NCBI_Bowtie2.fasta"
# Original path with all CpG coordinates
ORG_PATH = "/path/to/lab_storage/mouse_genome_annotation/Genes_Promoters_CpG_islands_for_Tae/allCpG_islands_GRCm39.bed" 

# Define column names for the input files
INPUT_COLUMNS = ["Chr1", "Start1", "End1", "Value", "MAPQ", "Chr2", "Start2", "End2"]

# Define columns for the output file
OUTPUT_COLUMNS = ["id", "sample", "chromosome", "strand", "GC_count", "damage", 
                  "GC_normalized_damage", "median", "median_normalized_damage"]

# Define the median values for each sample
# Median value for each sample from 100kb bins
# Read from a local file so measured values stay out of the repo, as {"sample_name": median, ...}
import json
with open("../data_normalized/sample_medians.json") as f:
    median_values = json.load(f)


In [ ]:
def run_bedtools_getfasta(bed_file, output_file, genome_path):
    """Run bedtools getfasta command to get sequence data for a BED file"""
    command = f"/cluster/software/stacks/2024-06/spack/opt/spack/linux-ubuntu22.04-x86_64_v3/gcc-12.2.0/bedtools2-2.31.0-a4obbslkxntgdx2criopqpwx662gcftq/bin/bedtools getfasta -fi {genome_path} -bed {bed_file} -bedOut > {output_file}"
    print(f"Running command: {command}")
    
    try:
        result = subprocess.run(command, shell=True, check=True, 
                               stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                               text=True)
        print("Command completed successfully")
        return True
    except subprocess.CalledProcessError as e:
        print(f"Error running bedtools: {e}")
        print(f"STDERR: {e.stderr}")
        return False

SEQ_COLUMNS = ["Chr", "Start", "End", "Seq"]
seq_output_file = os.path.join(PATH, "allCpG_islands.SEQ.bed")

# Run getfasta with original CpG coordinate file
run_bedtools_getfasta(ORG_PATH, seq_output_file, GENOME_PATH)

seq_df = pd.read_csv(seq_output_file, sep="\t", header=None, names=SEQ_COLUMNS)

seq_counts_plus = seq_df.copy()
seq_counts_plus['strand_id'] = seq_counts_plus['Chr'] + '+' + seq_counts_plus['Start'].astype(str)
seq_counts_plus['GC_count'] = seq_counts_plus['Seq'].str.upper().str.count('G')
seq_counts_plus = seq_counts_plus.drop_duplicates('strand_id')

seq_counts_minus = seq_df.copy()
seq_counts_minus['strand_id'] = seq_counts_minus['Chr'] + '-' + seq_counts_minus['Start'].astype(str)
seq_counts_minus['GC_count'] = seq_counts_minus['Seq'].str.upper().str.count('C')
seq_counts_minus = seq_counts_minus.drop_duplicates('strand_id')

# Combine both strand data
seq_counts = pd.concat([seq_counts_plus, seq_counts_minus])



In [ ]:
# Collect all files for each sample
bed_files = glob.glob(os.path.join(PATH, "*strand.bed"))

all_unique_ids = set()  # To store all unique IDs across all samples

# First pass: Collect all unique IDs across all samples
print("First pass: Collecting all unique IDs...")
for file_path in sorted(bed_files):

    try:
        # Read the file
        df = pd.read_csv(file_path, sep="\t", header=None, names=INPUT_COLUMNS)

        # Create the identifier
        ids = df['Chr1'] + "_" + df['Start2'].astype(str)
        
        # Add to the set of all unique IDs
        all_unique_ids.update(ids.unique())
        
    except Exception as e:
        print(f"  Error collecting IDs from {file_name}: {str(e)}")

print(f"Collected {len(all_unique_ids)} unique IDs across all samples")

# Create a dictionary to store chromosome info for each ID
id_to_chrom = {}
for file_path in sorted(bed_files):
  
    try:
        df = pd.read_csv(file_path, sep="\t", header=None, names=INPUT_COLUMNS)
        
        # Map each ID to its chromosome
        id_df = df.copy()

        id_df['id'] = id_df['Chr1'] + "_" + id_df['Start2'].astype(str)
        
        for _, row in id_df.drop_duplicates('id').iterrows():
            id_to_chrom[row['id']] = row['Chr1']
            
    except Exception as e:
        print(f"  Error mapping chromosomes: {str(e)}")

In [ ]:


print("Starting processing of samples")

# Process each file
sample_data_dict = {}  # Store data for each sample


for file_path in sorted(bed_files):
    file_name = os.path.basename(file_path)
    print(f"Processing file: {file_name}")
    
    # Extract sample name and determine strand
    sample_name = file_name.split('.')[0]
    
    if "plus_strand" in file_name:
        strand = "+"
    elif "minus_strand" in file_name:
        strand = "-"
    else:
        print(f"  Warning: Could not determine strand for {file_name}")
        continue
    
    try:
        # Read and process the file
        # dataframe of intersection from oxidation sites per sample and cpg coordinates
        df = pd.read_csv(file_path, sep="\t", header=None, names=INPUT_COLUMNS)
        print(f"  Records: {len(df)}")
        
        # Create IDs and calculate damage per genomic position
        df['strand_id'] = df['Chr1'] + strand + df['Start2'].astype(str)
        df['id'] = df['Chr1'] + "_" + df['Start2'].astype(str)
        
        damage_df = df.groupby('id').agg({
            'Value': 'sum',
            'Chr1': 'first',
            'strand_id': 'first'
        }).reset_index()
        damage_df = damage_df.rename(columns={'Value': 'damage'})
        damage_df = damage_df.rename(columns={'Chr1': 'chromosome'})
        
        # Create base result dataframe
        # We need one row per ID + strand + sample combination
        file_data = damage_df.copy()
        file_data['sample'] = sample_name
        file_data['strand'] = strand

        # 4. Merge the counts with our result dataframe
        file_data = file_data.merge(
            seq_counts[['strand_id', 'GC_count']], 
            on='strand_id', 
            how='left'
        )
        file_data.drop('strand_id', axis=1, inplace=True)
        
        # Calculate normalized damage
        file_data['GC_normalized_damage'] = file_data['damage'] / file_data['GC_count'].replace(0, np.nan)
        
        # Add median normalization if available
        if sample_name in median_values:
            file_data['median'] = median_values[sample_name]
            file_data['median_normalized_damage'] = 1000 * file_data['GC_normalized_damage'] / file_data['median']
        else:
            print(f"  Warning: No median value found for sample {sample_name}")
            file_data['median'] = np.nan
            file_data['median_normalized_damage'] = np.nan
        
        # Store processed data
        # file_data saved per sample
        sample_data_dict[f"{sample_name}_{strand}"] = file_data
        print(f"File data: {len(file_data)}")
        
        print (file_data.head(1))
        
    except Exception as e:
        print(f"  Error processing {file_name}: {str(e)}")




In [ ]:
# Fill in missing IDs for each sample
print("Adding missing IDs to each sample...")
for sample_key, file_data in sample_data_dict.items():

    
    sample_name = sample_key.rsplit('_', 1)[0]
    strand = sample_key.split('_')[-1]
    
    # Identify missing IDs for this sample
    existing_ids = set(file_data['id'])
    print (f"There are {len(existing_ids)} unique ids")
    missing_ids = all_unique_ids - existing_ids
    
    if missing_ids:
        print(f"  Adding {len(missing_ids)} missing IDs to sample {sample_name} (strand {strand})")
        
        # Create rows for missing IDs
        missing_rows = []
        for missing_id in missing_ids:
            # Get the chromosome for this ID
            chromosome = id_to_chrom.get(missing_id, "unknown")
            
            # Create a row with zeros for all numerical values
            missing_row = {
                'id': missing_id, 
                'sample': sample_name,
                'chromosome': chromosome,
                'strand': strand,
                'GC_count': 0,
                'damage': 0,
                'GC_normalized_damage': 0
            }
            
            # Add median value if available
            if sample_name in median_values:
                missing_row['median'] = median_values[sample_name]
                missing_row['median_normalized_damage'] = 0
            else:
                missing_row['median'] = np.nan
                missing_row['median_normalized_damage'] = np.nan
                
            missing_rows.append(missing_row)
        
        # Add the missing rows to the sample's data
        if missing_rows:
            missing_df = pd.DataFrame(missing_rows)
            print (missing_df.head(5))
            file_data = pd.concat([file_data, missing_df], ignore_index=True)
            sample_data_dict[sample_key] = file_data

       

In [ ]:
# Create an empty DataFrame for the final output
all_data = pd.DataFrame(columns=OUTPUT_COLUMNS)

# Combine all samples into the final DataFrame
for sample_key, file_data in sample_data_dict.items():
    all_data = pd.concat([all_data, file_data], ignore_index=True)

# Save the combined data to a CSV file
all_data.to_csv(OUTPUT_FILE, index=False)
print(f"Processing complete! Combined data saved to {OUTPUT_FILE}")
print(f"Total records: {len(all_data)}")